# 02 · Vectorization

No model is trained here. This notebook exists to make the vectorization
decisions *once*, with the counts in front of you, and write them into a
config that notebooks 03, 04 and 05 all load. Three models tuned against
three quietly different feature spaces are not comparable, and the
difference will look like a modelling result.

Three representations, three consumers:

| module | produces | used by |
|---|---|---|
| `vectorization.sparse` | BOW / TF-IDF matrix | 03, and any sklearn model on sparse input |
| `vectorization.vocab` | integer token indices | 04 (`EmbeddingBag`) |
| `vectorization.embeddings` | Word2Vec vectors, pooled | 04 (initialisation), 05 (features) |

One rule runs through all of it: **fit on the training split only**.
`fit_transform` is shaped so there is nowhere to pass the validation
texts that would cause them to be fitted on.

In [0]:
import sys, pathlib
# Repo root on the path, whether this runs from /notebooks locally or from a
# Databricks Repos checkout. Editable installs are avoided on purpose: the
# cluster does not keep them between restarts.
ROOT = pathlib.Path.cwd()
if not (ROOT / "nn_classifier").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

import nn_classifier as nnc
from nn_classifier import paths
print(paths.describe())


[paths] scratch: /local_disk0/tmp (cluster SSD): PermissionError -- using /tmp/nn_classifier
environment: Databricks
  data dir:   /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data
  output dir: /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs
  scratch:    /tmp/nn_classifier  (117 GB free)
  device:     cpu


In [0]:
from nn_classifier import data, vectorization
from nn_classifier.config import RunConfig, load_config

# 01's cleaning and filter choices, not a fresh default. This used
# to be RunConfig(), which meant the config saved here -- and with
# every model bundle downstream -- described a cleaning recipe that
# had not been run.
cfg = load_config(default=RunConfig())
print(f"filter_to_known_tokens = {cfg.cleaning.filter_to_known_tokens}")
df = data.load_data(cfg.data)
levels = list(cfg.data.level_columns)
train_df, val_df = data.prepare(df, cfg.data, cfg.filters, cfg.split)

[config] loaded /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/run_config.joblib
filter_to_known_tokens = False
[tokens] dropped 0 of 3,492 classes at 'Level 4 ID' seen < 6 times (0 rows, 0.00%)
[data] train=540,545 rows, validation=60,061 rows


## 1 · Sparse: BOW vs TF-IDF

The choice is not free-floating. TF-IDF down-weights terms that appear
everywhere, which is right for ordinary vocabulary and wrong for a vendor
marker — a marker on every one of a large vendor's rows gets a low IDF
weight, suppressing exactly the vendors there is most data for. So if
`attach_vendor` was on in notebook 01, use counts.

In [0]:
from nn_classifier.config import SparseVectorizerConfig

candidates = {
    "tfidf 1-gram":   SparseVectorizerConfig(kind="tfidf", ngram_range=(1, 1), min_df=2),
    "tfidf 1-2 gram": SparseVectorizerConfig(kind="tfidf", ngram_range=(1, 2), min_df=2),
    "tfidf unordered bigrams": SparseVectorizerConfig(
        kind="tfidf", ngram_range=(1, 2), min_df=2, unordered_bigrams=True),
    "count 1-gram":   SparseVectorizerConfig(kind="count", ngram_range=(1, 1), min_df=2),
    # What 03b actually settled on. Neutralising TF changed nothing
    # (3.5 tokens per description, so term frequency is 1 almost
    # everywhere), and dropping IDF and the L2 norm on top of that
    # also changed nothing -- BatchNorm after the first layer
    # absorbs the normalisation. It was the fastest of the three.
    "count binary 1-2 gram": SparseVectorizerConfig(
        kind="count", binary=True, ngram_range=(1, 2),
        min_df=3, unordered_bigrams=True),
}

rows = []
for name, c in candidates.items():
    vec, X_tr, (X_va,) = vectorization.fit_transform(
        c, train_df[cfg.data.text_col], val_df[cfg.data.text_col], verbose=False)
    rows.append({
        "config": name,
        "features": len(vec.vocabulary_),
        "density": float(X_tr.nnz / np.prod(X_tr.shape)),
        "mean_nonzero_per_row": float(X_tr.nnz / X_tr.shape[0]),
    })
pd.DataFrame(rows)

,config,features,density,mean_nonzero_per_row
0,tfidf 1-gram,16000,0.000256,4.089593
1,tfidf 1-2 gram,16000,0.000351,5.608804
2,tfidf unordered bigrams,16000,0.000355,5.675224
3,count 1-gram,16000,0.000256,4.089593
4,count binary 1-2 gram,16000,0.000355,5.675594


`mean_nonzero_per_row` is the number to watch. Product descriptions are
short, so a row with three or four non-zero features after cleaning means
the model is being asked to classify into thousands of categories from
almost nothing — and that is a cleaning problem, not a model problem.

It sat at ~3.5 for every run up to now, under the dictionary filter. With
that filter off it should rise sharply. **If it does not, stop here** —
the cleaning was not the bottleneck and there is no point training.

One caveat on this table: it is measured *before* token pruning, because
`min_token_count` is applied by the model notebooks, not here. So these
counts are an upper bound on what 03 will actually see. The number 03
prints is the real one.

Folding bigrams to an order-insensitive form is worth the small vocabulary
reduction it buys: a catalogue writes the same product both ways, and
treating `BALL VALVE` and `VALVE BALL` as two features halves the
evidence for each.

### Record the choice

Nothing above changes `cfg`. Pick the winning row from the table and
assign it here — otherwise notebook 03 trains against the dataclass
defaults while this notebook's comparison sits in the output, unused
and looking authoritative.

In [0]:
CHOSEN = "count binary 1-2 gram"   # <- set this from the table above

# Note this is recorded, not obeyed: notebook 03 overrides
# cfg.sparse in its own settings cell. Keeping the two in step
# matters for provenance, not for the run.

cfg.sparse = candidates[CHOSEN]
print(f"sparse vectorization: {CHOSEN}")
print(cfg.sparse)

sparse vectorization: count binary 1-2 gram
SparseVectorizerConfig(kind='count', max_features=16000, ngram_range=(1, 2), min_df=3, max_df=0.95, stop_words=None, lowercase=True, binary=True, sublinear_tf=True, norm='l2', use_idf=True, unordered_bigrams=True, extra={})


In [0]:
vec, X_train, (X_val,) = vectorization.fit_transform(
    cfg.sparse, train_df[cfg.data.text_col], val_df[cfg.data.text_col])

# X_train is passed on purpose. Without it this can only rank a TF-IDF
# vectorizer (via idf_) and cannot rank a CountVectorizer at all -- it
# used to quietly return an alphabetical slice instead, which looked
# like a frequency ranking and was not.
print(vectorization.top_features(vec, 40, X=X_train))

# Read `share` before `count`. A feature in 40% of the catalogue is a
# stopword in everything but name, however domain-specific it looks --
# and it is the number that should set max_df.
display(vectorization.feature_report(vec, X_train, n=40))

[vec] count vocabulary: 16,000 features (ngram_range=(1, 2), min_df=3)
[vec] X_train: (540545, 16000), density=0.00035
['drill', 'steel', 'high', 'glove', 'speed', 'high speed', 'socket', 'bit', 'speed steel', 'insert', 'dr', 'hex', 'carbide', 'resistant', 'national', 'endmill', 'bit drill', 'flame', 'flame resistant', 'jobber', 'wrench', 'hose', 'cut', 'tap', 'pipe', 'tape', 'disc', 'hd', 'screw', 'thread', 'set', 'metricsizedesignation', 'wire', 'punch', 'standard', 'impact', 'wheel', 'hand', 'kit', 'safety']


feature,documents,share
drill,33203.0,0.06142504324339324
steel,30185.0,0.05584178930523823
high,24368.0,0.04508042808646829
glove,22046.0,0.040784763525700915
speed,22022.0,0.040740363891997894
high speed,21489.0,0.03975432202684328
socket,20067.0,0.037123643729939225
bit,18297.0,0.03384917074434136
speed steel,16848.0,0.03116854285952141
insert,15537.0,0.028743212868493836


## 2 · Token vocabulary, for the dense network

`EmbeddingBag` takes token indices, not a matrix. Index 0 is always the
pad token and index 1 the unknown token — the model's `padding_idx=0` and
the empty-document guard in the collate function both depend on it.

In [0]:
vocab = vectorization.build_vocabulary(train_df[cfg.data.text_col], cfg.vocab)
print(f"validation OOV rate: {vocab.oov_rate(val_df[cfg.data.text_col]):.3%}")
print(vocab.itos[:20])

[vocab] 16,002 entries (16,000 tokens with min_df=2, 106,517 seen in total)
validation OOV rate: 5.958%
['<pad>', '<unk>', 'drill', 'steel', 'high', 'glove', 'speed', 'socket', 'bit', 'insert', 'dr', 'hex', 'carbide', 'resistant', 'national', 'endmill', 'flame', 'jobber', 'wrench', 'hose']


The OOV rate is the number to remember. It is the single most useful
diagnostic when scoring quality drops later without the code changing: a
rise here means the cleaning or the input distribution moved, not the
model.

## 3 · Word2Vec

Trained on the training descriptions only, and used two different ways
downstream — to initialise the dense network's embedding matrix
(notebook 04), and to build fixed document vectors for the sklearn models
(notebook 05).

**Skippable if you are only re-running 03.** Nothing in this section
is read by the sparse network — it feeds 04's embedding initialisation
and 05's document vectors. `retrain=True` below puts gensim on one
driver core over half a million descriptions, which is the slowest
cell in the notebook by a wide margin. Run cells 1–8, then jump to
section 4.

`backend="spark"` exists for when that becomes intolerable.

In [0]:
# Trained once here and saved; notebooks 04 and 05 load the same file.
# Word2Vec is stochastic, so retraining it per notebook would leave the
# dense network and the sklearn models sitting on different vectors, and
# part of any difference in their scores would be that.
wv = vectorization.load_or_train_word2vec(
    train_df[cfg.data.text_col], cfg.word2vec, retrain=True)
print(f"{len(wv):,} vectors of size {wv.vector_size}")

# Nearest neighbours are the cheapest sanity check there is. If the
# neighbours of a common product noun are not other product nouns, the
# corpus is too small or the cleaning is too aggressive.
import numpy as np
def neighbours(token, k=8):
    if token not in wv:
        return f"{token!r} not in the vocabulary"
    v = wv.vectors[token]
    sims = {t: float(vec @ v / (np.linalg.norm(vec) * np.linalg.norm(v) + 1e-12))
            for t, vec in wv.vectors.items() if t != token}
    return sorted(sims.items(), key=lambda kv: -kv[1])[:k]

for probe in ["valve", "glove", "drill", "bearing"]:
    print(probe, "->", neighbours(probe))

[deps] gensim not installed (needed by the gensim Word2Vec backend); installing now
[w2v] gensim trained 38,257 vectors of size 150
[w2v] saved to /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/word_vectors.joblib
38,257 vectors of size 150
valve -> [('alve', 0.7273743748664856), ('regulator', 0.625605046749115), ('cock', 0.6233474016189575), ('manifold', 0.6180222630500793), ('ono', 0.6115001440048218), ('1pce', 0.607445240020752), ('check', 0.6013057231903076), ('flow', 0.5930764675140381)]
glove -> [('glv', 0.84779292345047), ('mitt', 0.7669562101364136), ('endu', 0.7399227619171143), ('lining', 0.7300536036491394), ('2xs', 0.7258868217468262), ('palm', 0.7234752178192139), ('gaunt', 0.6819374561309814), ('vend', 0.6776707172393799)]
drill -> [('135d', 0.6914413571357727), ('thunderbit', 0.6264283657073975), ('118o', 0.6238406896591187), ('118deg', 0.6068952679634094), ('letter', 0.6028701066970825), ('parabolic', 0.5966488718986511)

In [0]:
X_train_dense, (X_val_dense,) = vectorization.build_document_matrices(
    wv, train_df[cfg.data.text_col], val_df[cfg.data.text_col], cfg=cfg.word2vec)
print(X_train_dense.shape, X_val_dense.shape)

[w2v] pooled 540,545 documents to 150 dims; 2,486 had no known tokens (0.46%)
[w2v] pooled 60,061 documents to 150 dims; 333 had no known tokens (0.55%)
(540545, 150) (60061, 150)


## 4 · Save the decisions

Everything settled here — the vectorizer, the token vocabulary, the
Word2Vec geometry, and the shared data/cleaning/split settings — is
written to `outputs/run_config.joblib`. Notebooks 03, 04 and 05 load
that file rather than defining their own.

What is deliberately *not* settled here: the model hyperparameters.
`cfg.nn` and `cfg.sklearn` belong to the individual model notebooks,
each of which overrides them locally without saving. A shared file that
every notebook wrote to would mean the last one run silently redefined
the others.

In [0]:
# The shared front-half settings, in one place, before saving.
cfg.vocab.vocab_size = 16000
cfg.vocab.min_df = 2

cfg.word2vec.vector_size = 150
cfg.word2vec.window = 5
cfg.word2vec.min_count = 2
cfg.word2vec.epochs = 5
cfg.word2vec.backend = "gensim"      # "spark" on a large catalogue

cfg.hierarchy.joint_mode = "independent"   # "conditional" | None
cfg.hierarchy.predict_all_levels = True

# THE setting this notebook exists to carry. 03/04/05 read it out
# of the saved config and call drop_one_off_tokens with it, counting
# on the training split only. Nothing in notebook 01 applies it, so
# setting it there does nothing.
#
# 5, replacing the dictionary filter that 01 now has switched off.
# A token appearing in five or more training descriptions is
# evidence whatever NLTK thinks of it; one seen once or twice is a
# part number that identifies a row without teaching anything about
# the next one.
cfg.filters.min_token_count = 5

# Left at 0, and it is not the lever it sounds like: it keeps tokens
# appearing under AT LEAST this many classes -- a breadth filter.
# What would be useful here is the opposite, keeping tokens
# CONCENTRATED in few classes, and that does not exist yet.
cfg.filters.min_token_classes = 0

print(cfg.describe())

{
  "data": {
    "path": "/Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data/cleaned_data.txt",
    "file_format": "csv",
    "delimiter": "|",
    "text_col": "FullDesc",
    "id_col": "ID",
    "vendor_col": "Vallen ID",
    "level_columns": [
      "Level 1 ID",
      "Level 2 ID",
      "Level 3 ID",
      "Level 4 ID"
    ],
    "name_columns": [
      "Level 1 Name",
      "Level 2 Name",
      "Level 3 Name",
      "Level 4 Name"
    ],
    "drop_null_text": true,
    "drop_null_labels": true
  },
  "cleaning": {
    "strip_leading_code": true,
    "uppercase": false,
    "normalize_misspellings": true,
    "expand_abbrevs": true,
    "filter_to_known_tokens": false,
    "keep_abbrev_keys": true,
    "min_token_length": 2,
    "attach_vendor": false,
    "vendor_token_prefix": "VND_"
  },
  "filters": {
    "min_class_count": 6,
    "min_path_count": 2,
    "min_token_count": 5,
    "min_token_classes": 0,
    "ngram_range_for_counts":

In [0]:
from nn_classifier.config import save_config

cfg.validate()          # catches the mistakes that do not raise on their own
save_config(cfg)        # word_vectors.joblib was already written above
print(cfg.describe()[:1200])

[config] wrote /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/run_config.joblib and /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/run_config.json
{
  "data": {
    "path": "/Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data/cleaned_data.txt",
    "file_format": "csv",
    "delimiter": "|",
    "text_col": "FullDesc",
    "id_col": "ID",
    "vendor_col": "Vallen ID",
    "level_columns": [
      "Level 1 ID",
      "Level 2 ID",
      "Level 3 ID",
      "Level 4 ID"
    ],
    "name_columns": [
      "Level 1 Name",
      "Level 2 Name",
      "Level 3 Name",
      "Level 4 Name"
    ],
    "drop_null_text": true,
    "drop_null_labels": true
  },
  "cleaning": {
    "strip_leading_code": true,
    "uppercase": false,
    "normalize_misspellings": true,
    "expand_abbrevs": true,
    "filter_to_known_tokens": false,
    "keep_abbrev_keys": tr